# Theory reconstruction and validation

This notebook interrogates the installed `desilike` / Velocileptors implementation and tests whether its PT component table can reconstruct the final multipoles. It is deliberately separate from emulator training: arbitrary sample cosmologies are allowed here.

**Success criterion:** an independently assembled spectrum agrees with the native theory pipeline to numerical precision on the same k grid. Do not proceed to emulator training on the assumption that a visually close match is sufficient.

In [ ]:
from pathlib import Path
import sys, inspect, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)


## 1. Locate the project and load sample cosmologies

The expected NERSC checkout is `/global/u1/z/zbrown5/power_emulator/project`. The catalogue path is discovered rather than silently assumed; set `COSMOLOGY_CSV` manually if your checkout stores it elsewhere.

In [ ]:
from pathlib import Path
import sys, inspect, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# EDIT THESE TWO PATHS for your NERSC checkout and cosmology catalogue.
# PROJECT_ROOT must contain theory.py, cosmology.py, and nuisance.py.
PROJECT_ROOT = Path('/path/to/project')
COSMOLOGY_CSV = Path('/path/to/cosmologies.csv')

if not (PROJECT_ROOT / 'theory.py').exists():
    raise FileNotFoundError(f'Could not find theory.py under PROJECT_ROOT={PROJECT_ROOT}. Edit the path above.')
if not COSMOLOGY_CSV.exists():
    raise FileNotFoundError(f'Could not find cosmology catalogue: {COSMOLOGY_CSV}. Edit the path above.')

sys.path.insert(0, str(PROJECT_ROOT.parent))
print('PROJECT_ROOT:', PROJECT_ROOT)
print('COSMOLOGY_CSV:', COSMOLOGY_CSV)
print('Python:', sys.version)

from project.cosmology import read_abacus_cosmologies, get_cosmology_dict, get_cosmology_metadata
from project.theory import TheoryManager, K_FID, ELL_FID, BIAS_PARAMETERS, EFT_PARAMETERS, STOCHASTIC_PARAMETERS
from project.nuisance import get_nuisance_dict

cosmologies = read_abacus_cosmologies(COSMOLOGY_CSV, cosmology_ids=[130])
print('\nSelected cosmology:')
display(cosmologies)
cosmo = get_cosmology_dict(cosmologies.iloc[0])
cosmo_meta = get_cosmology_metadata(cosmologies.iloc[0])
print('\nIndependent cosmological parameters:', json.dumps(cosmo, indent=2))
print('Reference / bookkeeping metadata:', json.dumps(cosmo_meta, indent=2))


## 2. Initialize the theory and inspect its internal representation

The first evaluation forces initialization. We then print object attributes, array shapes, and relevant method signatures. In particular, verify the reported PT table dimensions and discover the actual k coordinate and component labels from the installed version rather than relying on assumptions.

In [ ]:
manager = TheoryManager(max_cache_size=4)
baseline_nuisance = {
    'bias': {'b1': 2.0, 'b2': 0.0, 'bs': 0.0, 'b3': 0.0},
    'eft': {'alpha0': 0.0, 'alpha2': 0.0, 'alpha4': 0.0, 'alpha6': 0.0},
    'stochastic': {'sn0': 0.0, 'sn2': 0.0, 'sn4': 0.0},
}
P_baseline = manager.evaluate(cosmo, **baseline_nuisance)
theory, pipe = manager.get_theory(cosmo)

print('Theory class:', type(theory))
print('Pipeline class:', type(pipe))
print('Output shape:', P_baseline.shape, ' expected:', (len(ELL_FID), len(K_FID)))
print('Output finite:', np.isfinite(P_baseline).all())
print('k grid:', len(K_FID), 'bins; range', (K_FID.min(), K_FID.max()))
print('ells:', ELL_FID)
print('\nTheory public-ish attributes:')
for name in sorted(n for n in dir(theory) if not n.startswith('__')):
    try:
        value = getattr(theory, name)
        if isinstance(value, (str, int, float, bool, tuple, list, np.ndarray)):
            if isinstance(value, np.ndarray): detail = f'array shape={value.shape}, dtype={value.dtype}'
            else: detail = repr(value)[:180]
            print(f'  {name}: {detail}')
    except Exception as exc:
        print(f'  {name}: <access raised {type(exc).__name__}>')

pt = getattr(theory, 'pt', None)
print('\nPT object:', type(pt))
if pt is not None:
    for name, value in vars(pt).items():
        if isinstance(value, np.ndarray):
            print(f'  pt.{name}: shape={value.shape}, dtype={value.dtype}, finite={np.isfinite(value).all()}')
        else:
            print(f'  pt.{name}: {type(value).__name__} = {str(value)[:160]}')


## 3. Inspect the implementation that combines PT components

Use Python introspection on the installed package. This is important because component ordering and nuisance-to-component coefficients are implementation details that can change between `desilike` versions.

In [ ]:
print('Installed desilike version:')
try:
    import desilike
    print(getattr(desilike, '__version__', '<version attribute unavailable>'))
except Exception as exc: print('Could not read version:', repr(exc))

classes_to_inspect = [type(theory)]
if pt is not None: classes_to_inspect.append(type(pt))
for cls in classes_to_inspect:
    print('\n' + '='*88 + '\nCLASS:', cls.__module__ + '.' + cls.__name__)
    for name, member in inspect.getmembers(cls, predicate=inspect.isfunction):
        key = name.lower()
        if any(token in key for token in ('calculate', 'combine', 'table', 'spectrum', 'initialize', 'setup')):
            print(f'\n--- {name}{inspect.signature(member)} ---')
            try: print(inspect.getsource(member)[:12000])
            except Exception as exc: print('<source unavailable:', repr(exc), '>')


## 4. Nuisance response diagnostics

Vary each sampled nuisance coefficient while holding the others at the fiducial point. `b3` and `alpha6` remain zero in this first pass; `fsat`, `sigv`, and `rvir` remain at the LRG-like values supplied by `TheoryManager`.

In [ ]:
scan_values = {
    'b1': [1.5, 2.0, 2.5], 'b2': [-2.0, 0.0, 2.0], 'bs': [-2.0, 0.0, 2.0],
    'alpha0': [-20.0, 0.0, 20.0], 'alpha2': [-20.0, 0.0, 20.0], 'alpha4': [-20.0, 0.0, 20.0],
    'sn0': [-0.5, 0.0, 0.5], 'sn2': [-10.0, 0.0, 10.0], 'sn4': [-300.0, 0.0, 300.0],
}
parameter_group = {name: group for group, names in [('bias', BIAS_PARAMETERS), ('eft', EFT_PARAMETERS), ('stochastic', STOCHASTIC_PARAMETERS)] for name in names}
response_summary = []
scan_spectra = {}
for name, values in scan_values.items():
    group = parameter_group[name]
    for value in values:
        nuisance = {key: dict(val) for key, val in baseline_nuisance.items()}
        nuisance[group][name] = float(value)
        P = manager.evaluate(cosmo, **nuisance)
        scan_spectra[(name, value)] = P
        delta = P - P_baseline
        response_summary.append({
            'parameter': name, 'value': value,
            'max_abs_delta': float(np.max(np.abs(delta))),
            'rms_delta': float(np.sqrt(np.mean(delta**2))),
            'finite': bool(np.isfinite(P).all()),
        })
response_df = pd.DataFrame(response_summary)
display(response_df)
print('\nCache diagnostics:', manager.cache_info())


## 5. Test whether the PT table spans the native spectrum

This is a deliberately conservative first reconstruction test. It extracts the actual `pt.table`, identifies its k coordinate if exposed, and attempts a least-squares projection of the native output onto the table's component axis. The fit is a **span test**, not yet proof that the fitted coefficients are the physically correct nuisance-coefficient map. The previous cell's source inspection is what tells us how to construct that map.

In [ ]:
table = getattr(pt, 'table', None) if pt is not None else None
if table is None:
    print('No pt.table attribute found; inspect the PT object dump and source above.')
else:
    table = np.asarray(table)
    print('pt.table shape:', table.shape)
    print('pt.table finite:', np.isfinite(table).all())
    print('PT object k-like attributes:')
    for name, value in vars(pt).items():
        if 'k' in name.lower() and hasattr(value, 'shape'):
            print(' ', name, np.shape(value), np.asarray(value).reshape(-1)[:8])

    # Locate a one-dimensional k coordinate with the table's second axis length.
    n_internal = table.shape[1] if table.ndim == 3 else None
    k_internal = None
    if n_internal is not None:
        for name, value in vars(pt).items():
            arr = np.asarray(value) if isinstance(value, (np.ndarray, list, tuple)) else None
            if arr is not None and arr.ndim == 1 and arr.size == n_internal and 'k' in name.lower():
                k_internal = arr.astype(float)
                print('Using internal k coordinate:', name)
                break

    if table.ndim != 3:
        print('Unexpected table rank; do not infer component axis ordering.')
    elif k_internal is None:
        print('Could not identify the table k coordinate automatically. No projection attempted.')
    elif table.shape[0] != len(ELL_FID):
        print('Table multipole axis does not match requested ells; inspect source before proceeding.')
    else:
        # Interpolate the native pipeline result onto the table's internal k grid.
        # np.interp requires increasing coordinates.
        order = np.argsort(K_FID)
        P_internal = np.vstack([np.interp(k_internal, K_FID[order], row[order]) for row in P_baseline])
        projection_coefficients = []
        projection_reconstruction = np.zeros_like(P_internal)
        for iell in range(table.shape[0]):
            # Candidate interpretation: table[ell, k, component].
            design = table[iell]
            coeff, *_ = np.linalg.lstsq(design, P_internal[iell], rcond=None)
            projection_coefficients.append(coeff)
            projection_reconstruction[iell] = design @ coeff
        projection_coefficients = np.asarray(projection_coefficients)
        residual = projection_reconstruction - P_internal
        print('Projection coefficient array:', projection_coefficients.shape)
        print('Span-test max abs residual:', np.max(np.abs(residual)))
        print('Span-test relative RMS residual:', np.sqrt(np.mean(residual**2)) / max(np.sqrt(np.mean(P_internal**2)), 1e-30))
        print('NOTE: this projection is diagnostic only; coefficients are not yet identified with nuisance parameters.')


## 6. Multiple cosmologies and cache behavior

Once the first cosmology passes, add more IDs here. This verifies that the PT table changes with cosmology while repeated nuisance evaluations at a fixed cosmology reuse the cached theory object.

In [ ]:
EXTRA_COSMOLOGY_IDS = []  # e.g. [130,  ...]; start with c130 until the first reconstruction is understood.
if EXTRA_COSMOLOGY_IDS:
    extra = read_abacus_cosmologies(COSMOLOGY_CSV, cosmology_ids=EXTRA_COSMOLOGY_IDS)
    multi_cosmo_results = []
    for _, row in extra.iterrows():
        c = get_cosmology_dict(row)
        P = manager.evaluate(c, **baseline_nuisance)
        t, _ = manager.get_theory(c)
        tab = np.asarray(t.pt.table)
        multi_cosmo_results.append({'cosmo_id': int(row.cosmo_id), 'P_shape': P.shape, 'table_shape': tab.shape,
                                    'P_rms': float(np.sqrt(np.mean(P**2))), 'table_rms': float(np.sqrt(np.mean(tab**2)))})
    display(pd.DataFrame(multi_cosmo_results))
else:
    print('No additional cosmology IDs selected yet; c130 remains the diagnostic anchor.')

print('Final cache diagnostics:', manager.cache_info())


## 7. Human-readable power-spectrum plots

These are actual outputs from the installed theory pipeline, not mock curves. The first figure shows the fiducial spectra; the second shows selected one-at-a-time nuisance responses relative to the fiducial model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
for i, (ell, ax) in enumerate(zip(ELL_FID, axes)):
    ax.plot(K_FID, P_baseline[i], lw=2, label=fr'$P_{{{ell}}}$')
    ax.set_title(fr'Multipole $\ell={ell}$')
    ax.set_xlabel(r'$k\ [h\,\mathrm{{Mpc}}^{{-1}}]$')
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_ylabel(r'$P_\ell(k)\ [ (h^{-1}\,\mathrm{Mpc})^3 ]$')
fig.suptitle('c130 fiducial LPT–Velocileptors power spectra', y=1.03)
fig.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True)
examples = [('b1', 2.5), ('alpha0', 20.0), ('sn0', 0.5)]
for i, (ell, ax) in enumerate(zip(ELL_FID, axes)):
    for name, value in examples:
        delta = scan_spectra[(name, value)][i] - P_baseline[i]
        ax.plot(K_FID, delta, lw=1.8, label=f'{name}={value:g}')
    ax.axhline(0.0, color='black', lw=0.8)
    ax.set_title(fr'$\Delta P_{{{ell}}}(k)$')
    ax.set_xlabel(r'$k\ [h\,\mathrm{{Mpc}}^{{-1}}]$')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
axes[0].set_ylabel(r'$P_\ell - P_{\ell,\mathrm{fid}}$')
fig.suptitle('Example nuisance responses around the c130 fiducial model', y=1.03)
fig.tight_layout()
plt.show()


## 8. Validation checklist

Do not mark the factorization as complete until all of the following are true:

- [ ] `pt.table` axis ordering, internal k coordinate, and component labels are established from the installed implementation.
- [ ] The nuisance-to-component coefficient map is derived from source/implementation, not inferred from a single least-squares fit.
- [ ] A standalone reconstruction from cosmology-dependent components and nuisance coefficients matches the native pipeline at numerical precision for c130.
- [ ] The same reconstruction works for at least two additional cosmologies and several joint nuisance settings.
- [ ] The reconstruction is independent of pipeline evaluation after the component table and coefficients have been captured.

Record the maximum absolute and relative residuals for every test. A previous implementation showed a discrepancy of roughly 0.56%; that is not an acceptable final result.